## The State of Tax Justice: Estimate misalignment
- Author: Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 August 2023
- Last updated: 22 September 2024

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
- Details on the misalignment method and its background can be found here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455, the working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 
- This notebook estimates profit misalignment based on different formulas. It uses the dataset "data/final/cbcr_main.csv" (for the estimation with imputed values) or the dataset "data/final/cbcr_main_noimputation_allsubgroupsonly.csv" (for the estimation without imputed values). 

**Outline**
1. 
2. 
3. 

**To dos before running this notebook**
1. Run the notebooks 1_clean and 2_imupte_missings. Note the requirements of these notebooks. 

### 0. Load packages

In [1]:
import pandas as pd
import numpy as np
import tjn_tools
from config import *

In [2]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')
# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]
# Exclude countries that do no not report on an actual country by country basis (see list of reporting countries from notebook 1_clean below)
countries_to_exclude = ['AUT', 'CZE', 'FIN', 'GBR', 'HUN', 'IMN', 'IRL', 'KOR', 'MAC', 'MAR', 'MUS', 'NZL', 'SWE']
# Drop rows where 'iso_parent' is in the list of countries to exclude
cbcr_sample = cbcr_sample[~cbcr_sample['iso_parent'].isin(countries_to_exclude)]
# Replace missing ETRs (only two in sample) with CIT
cbcr_sample['etr_average_corrected'] = cbcr_sample['etr_average_corrected'].fillna(cbcr_sample['cit'])

### 1. Define misalignment

In [3]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_misalignment).reset_index(drop=True)

    return cbcr_data


### 2. Calculate misalignment for sample with full information

#### 2.1 Import data

Import data without imputed values. This is only the data from the sample of reporting countries that actually is reported on a country basis, i.e. excluding aggregated country groups and data from reporting countries that do not report on a country by country basis, but just by continents.
- In the 2024 data, the following reporting countries do not report country-by-country: 
    - Austria: Only continents
    - Czechia: Only Czechia versus rest of the world
    - Finland: Only Finland, continents and "other" for each continent (e.g. "Asia" and "Other Asia")
    - United Kingdom: Only UK and continents
    - Hungary: Only Hungary versus rest of the world
    - Isle of Man: Only continents
    - Ireland: Only Ireland versus rest of the world
    - Korea: Only Korea and continents
    - Macao: Only Macao versus rest of the world
    - Morocco: Only Morocco and continents
    - Mauritius: Only Mauritius and continents
    - New Zealand: Only New Zealand versus rest of the world
    - Sweden: Only Sweden, continents and "other" for each continent (e.g. "Asia" and "Other Asia")

In [4]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')
# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]
# Exclude countries that do no not report on an actual country by country basis (see list of reporting countries from notebook 1_clean below)
countries_to_exclude = ['AUT', 'CZE', 'FIN', 'GBR', 'HUN', 'IMN', 'IRL', 'KOR', 'MAC', 'MAR', 'MUS', 'NZL', 'SWE']
# Drop rows where 'iso_parent' is in the list of countries to exclude
cbcr_sample = cbcr_sample[~cbcr_sample['iso_parent'].isin(countries_to_exclude)]
# Replace missing ETRs (only two in sample) with CIT
cbcr_sample['etr_average_corrected'] = cbcr_sample['etr_average_corrected'].fillna(cbcr_sample['cit'])

#### 2.2 Calculate misalignment for sample countries with full information


In [5]:
# Initialize a list to store the aggregate results
results_sample = []

for year in range(first_year, first_year + n_years):
    print(f"Total profit shifted in USD mn {year}")
    
    misalignment_year = cbcr_sample[cbcr_sample['year'] == year].copy()
    misalignment_year = calculate_misalignment(misalignment_year, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

    # Keep only the first occurrence of these unique variables for each 'iso_partner'
    unique_columns = misalignment_year.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

    # Perform the groupby operation on 'iso_partner'
    country_results_year = misalignment_year.groupby(['iso_partner']).agg(
        negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        theoretical_profit=('theoretical_profit', 'sum'),
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
    ).reset_index()

    # Convert results to millions
    country_results_year['negative_misalignment'] = -country_results_year['negative_misalignment'] / 1e6
    country_results_year['positive_misalignment'] = country_results_year['positive_misalignment'] / 1e6
    country_results_year['theoretical_profit'] = country_results_year['theoretical_profit'] / 1e6
    country_results_year['reported_profit'] = country_results_year['reported_profit'] / 1e6

    # Merge the unique columns back into the result
    country_results_year = country_results_year.merge(unique_columns, on='iso_partner', how='left')

    # Calculate other relevant variables
    country_results_year['tax_revenue_loss'] = country_results_year['negative_misalignment'] * country_results_year['cit']
    country_results_year['tax_revenue_gain'] = country_results_year['positive_misalignment'] * country_results_year['etr_average_corrected']

    country_results_year['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
        country_results_year['gvt_health_expenditure'] == 0, 
        np.nan, 
        country_results_year['tax_revenue_loss'] / (country_results_year['gvt_health_expenditure'] / 1e6)
    )
    
    country_results_year['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
        country_results_year['tax_revenue_current_usd'] == 0, 
        np.nan, 
        country_results_year['tax_revenue_loss'] / (country_results_year['tax_revenue_current_usd'] / 1e6)
    )

    # Calculate totals
    total_positive_misalignment = country_results_year['positive_misalignment'].sum()
    total_negative_misalignment = country_results_year['negative_misalignment'].sum()
    total_tax_revenue_loss = country_results_year['tax_revenue_loss'].sum()
    total_tax_revenue_gain = country_results_year['tax_revenue_gain'].sum()
    average_tax_revenue_loss_pct_of_gvt_health_expenditure = country_results_year['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
    average_tax_revenue_loss_pct_of_total_tax_revenues = country_results_year['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

    print(f"Year {year}: Positive Misalignment: {total_positive_misalignment}, Negative Misalignment: {total_negative_misalignment}, "
          f"Total tax revenue loss: {total_tax_revenue_loss}, Total tax revenue gain: {total_tax_revenue_gain}")

    # Calculate countries' fractions of totals
    country_results_year['tax_revenue_loss_caused_pct_of_total'] = country_results_year['positive_misalignment'] / total_positive_misalignment
    country_results_year['tax_revenue_loss_caused_usd'] = country_results_year['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
    country_results_year['tax_revenue_loss_suffered_pct_of_total'] = country_results_year['tax_revenue_loss'] / total_tax_revenue_loss

    country_results_year = country_results_year[['iso_partner', 'partner_jurisdiction', 'negative_misalignment',
       'tax_revenue_loss', 'tax_revenue_loss_suffered_pct_of_total', 'tax_revenue_loss_pct_of_gvt_health_expenditure',
       'tax_revenue_loss_pct_of_total_tax_revenues', 'positive_misalignment', 'tax_revenue_gain', 
       'tax_revenue_loss_caused_usd', 'tax_revenue_loss_caused_pct_of_total', 'etr_average_corrected', 'cit',
       'region_tjn', 'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

    country_results_year = country_results_year.sort_values(by='iso_partner')
    country_results_year.to_csv(f'{output_tables}/SOTJ_{year}.csv', index=False)
    
    # Append aggregate results to the list
    results_sample.append({
        'year': year,
        'total_positive_misalignment': total_positive_misalignment,
        'total_negative_misalignment': total_negative_misalignment,
        'total_tax_revenue_loss': total_tax_revenue_loss,
        'total_tax_revenue_gain': total_tax_revenue_gain,
        'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure,
        'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues
    })

# Convert aggregate results to a DataFrame
results_sample_df = pd.DataFrame(results_sample)

# Save the aggregated results to a CSV or Excel file
results_sample_df.to_csv(f'{output_tables}/SOTJ_results_sample.csv', index=False)

Total profit shifted in USD mn 2016
Year 2016: Positive Misalignment: 391467.68202753784, Negative Misalignment: 391467.68202753784, Total tax revenue loss: 107773.00793744676, Total tax revenue gain: 23044.79644414011
Total profit shifted in USD mn 2017
Year 2017: Positive Misalignment: 785578.8511043687, Negative Misalignment: 785578.8511043687, Total tax revenue loss: 214290.18429793758, Total tax revenue gain: 47105.425422647175
Total profit shifted in USD mn 2018
Year 2018: Positive Misalignment: 704731.1823134198, Negative Misalignment: 704731.1823134199, Total tax revenue loss: 187756.303898895, Total tax revenue gain: 45671.127332278134
Total profit shifted in USD mn 2019
Year 2019: Positive Misalignment: 899347.0075299746, Negative Misalignment: 899347.0075299747, Total tax revenue loss: 234547.78008275953, Total tax revenue gain: 65580.36377680769
Total profit shifted in USD mn 2020
Year 2020: Positive Misalignment: 817923.3330477182, Negative Misalignment: 817923.3330477182,

### 3. Calculate misalignment for sample with imputed values
- This means that we calculate misalignment for each of the imputed datasets.
- After doing this, we take the median and the 5 and 95 percentiles of the estimated values to get a range of results

#### 3.1 Import data

In [7]:
cbcr_full = pd.read_csv(f'{data_final}/cbcr_with_imputed_data.csv')

C:\Users\AlisonSchultz\AppData\Local\Temp\ipykernel_10024\3764691491.py:1: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  cbcr_full = pd.read_csv(f'{data_final}/cbcr_with_imputed_data.csv')


#### 3.2 Calculate misalignment for samples with imputed values and aggregate results

In [8]:
# Initialize lists to store results
all_replications = []
aggregate_results = []

# Iterate over each imputation replication and year
for year in range(first_year, first_year + n_years):
    for rep, data in cbcr_full.groupby("n_rep"):
        print(f"Processing year {year}, replication {rep}")
        
        # Filter the data for the current year and replication
        misalignment_year = data[data['year'] == year].copy()
        misalignment_year = calculate_misalignment(misalignment_year, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

        # Keep only the first occurrence of these unique variables for each 'iso_partner'
        unique_columns = misalignment_year.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                                   'etr_average_corrected', 'cit',
                                                                                   'tax_revenue_current_usd', 
                                                                                   'gvt_health_expenditure', 'region_tjn', 
                                                                                   'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

        # Perform the groupby operation on 'iso_partner'
        country_results_year = misalignment_year.groupby(['iso_partner']).agg(
            negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
            positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
            theoretical_profit=('theoretical_profit', 'sum'),
            reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
        ).reset_index()

        # Convert results to millions
        country_results_year['negative_misalignment'] = -country_results_year['negative_misalignment'] / 1e6
        country_results_year['positive_misalignment'] = country_results_year['positive_misalignment'] / 1e6
        country_results_year['theoretical_profit'] = country_results_year['theoretical_profit'] / 1e6
        country_results_year['reported_profit'] = country_results_year['reported_profit'] / 1e6

        # Merge the unique columns back into the result
        country_results_year = country_results_year.merge(unique_columns, on='iso_partner', how='left')

        # Calculate other relevant variables
        country_results_year['tax_revenue_loss'] = country_results_year['negative_misalignment'] * country_results_year['cit']
        country_results_year['tax_revenue_gain'] = country_results_year['positive_misalignment'] * country_results_year['etr_average_corrected']

        country_results_year['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
            country_results_year['gvt_health_expenditure'] == 0, 
            np.nan, 
            country_results_year['tax_revenue_loss'] / (country_results_year['gvt_health_expenditure'] / 1e6)
        )

        country_results_year['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
            country_results_year['tax_revenue_current_usd'] == 0, 
            np.nan, 
            country_results_year['tax_revenue_loss'] / (country_results_year['tax_revenue_current_usd'] / 1e6)
        )

        # Add replication number and year
        country_results_year['year'] = year
        country_results_year['n_rep'] = rep

        # Append the replication results to the list
        all_replications.append(country_results_year)

# Concatenate all the results from the replications into a single DataFrame
df_all_replications = pd.concat(all_replications, ignore_index=True)

# Calculate summary statistics (mean, median, std, percentiles)
country_results_bootstrapped = df_all_replications.groupby(['iso_partner', 'year']).agg(
    median_positive_misalignment=('positive_misalignment', 'median'),
    median_negative_misalignment=('negative_misalignment', 'median'),
    avg_positive_misalignment=('positive_misalignment', 'mean'),
    avg_negative_misalignment=('negative_misalignment', 'mean'),
    sd_positive_misalignment=('positive_misalignment', 'std'),
    sd_negative_misalignment=('negative_misalignment', 'std'),
    percentile_2_5_positive_misalignment=('positive_misalignment', lambda x: x.quantile(0.025)),
    percentile_2_5_negative_misalignment=('negative_misalignment', lambda x: x.quantile(0.025)),
    percentile_97_5_positive_misalignment=('positive_misalignment', lambda x: x.quantile(0.975)),
    percentile_97_5_negative_misalignment=('negative_misalignment', lambda x: x.quantile(0.975))
).reset_index()

# Add the actual data from 'cbcr_sample' to replace imputed data
country_results_bootstrapped = country_results_bootstrapped.merge(
    cbcr_sample[['iso_partner', 'year', 'gdp_current_usd', 'gvt_health_expenditure', 'population', 'cit', 'etr_average_corrected']],
    on=['iso_partner', 'year'],
    how='left'
)

# Global aggregates for total results across all countries
country_results_bootstrapped['total_avg_positive_misalignment'] = country_results_bootstrapped.groupby('year')['avg_positive_misalignment'].transform('sum')
country_results_bootstrapped['total_avg_negative_misalignment'] = country_results_bootstrapped.groupby('year')['avg_negative_misalignment'].transform('sum')

# Tax revenue loss and gain based on CIT and ETR
country_results_bootstrapped["loss_incurred_cit_usd"] = country_results_bootstrapped["avg_negative_misalignment"] * country_results_bootstrapped["cit"]
country_results_bootstrapped["loss_incurred_etr_usd"] = country_results_bootstrapped["avg_negative_misalignment"] * country_results_bootstrapped["etr_average_corrected"]

# Total losses per year (CIT and ETR)
country_results_bootstrapped["total_loss_cit"] = country_results_bootstrapped.groupby('year')["loss_incurred_cit_usd"].transform('sum')
country_results_bootstrapped["total_loss_etr"] = country_results_bootstrapped.groupby('year')["loss_incurred_etr_usd"].transform('sum')

# Calculate shares of losses and inflicted losses based on misalignment
country_results_bootstrapped["loss_inflicted_share"] = country_results_bootstrapped["avg_positive_misalignment"] / country_results_bootstrapped["total_avg_positive_misalignment"]
country_results_bootstrapped["loss_inflicted_cit_usd"] = country_results_bootstrapped["loss_inflicted_share"] * country_results_bootstrapped["total_loss_cit"]
country_results_bootstrapped["loss_inflicted_etr_usd"] = country_results_bootstrapped["loss_inflicted_share"] * country_results_bootstrapped["total_loss_etr"]

# Save the final dataset
country_results_bootstrapped.to_csv(f'{output_tables}/country_results_bootstrapped.csv', index=False)

# Aggregate results across all countries for the global output
aggregate_df = country_results_bootstrapped.groupby('year').agg(
    total_positive_misalignment=('total_avg_positive_misalignment', 'mean'),
    total_negative_misalignment=('total_avg_negative_misalignment', 'mean'),
    total_loss_cit=('total_loss_cit', 'mean'),
    total_loss_etr=('total_loss_etr', 'mean')
).reset_index()

# Save global aggregate results
aggregate_df.to_csv(f'{output_tables}/global_aggregate_results.csv', index=False)

Processing year 2016, replication 0.0


KeyError: 'iso_parent'